<a href="https://colab.research.google.com/github/vaideheedaf/project-practice/blob/main/sih.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files

uploaded = files.upload()

Saving visdrone.ndjson to visdrone.ndjson


In [2]:
from google.colab import files

uploaded = files.upload()

Saving sard.ndjson to sard.ndjson


In [3]:
import json
import os
import requests
from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# ============================================================
# CONFIGURATION
# ============================================================

BASE = Path("/content/combined_object_detection")

for split in ["train", "val", "test"]:
    (BASE / "images" / split).mkdir(parents=True, exist_ok=True)
    (BASE / "labels" / split).mkdir(parents=True, exist_ok=True)

print("✅ Dataset folders created")


# ============================================================
# FINAL UNIFIED CLASS MAPPING
# ============================================================

CLASS_MAP = {
    "person": 0,
    "pedestrian": 0,
    "people": 0,
    "bicycle": 1,
    "car": 2,
    "van": 3,
    "truck": 4,
    "tricycle": 5,
    "awning-tricycle": 6,
    "bus": 7,
    "motor": 8
}

CLASS_NAMES = [
    "person",
    "bicycle",
    "car",
    "van",
    "truck",
    "tricycle",
    "awning-tricycle",
    "bus",
    "motor"
]


# ============================================================
# DATASETS
# ============================================================

DATASETS = [
    {
        "file": "/content/sard.ndjson",
        "name": "SARD"
    },
    {
        "file": "/content/visdrone.ndjson",
        "name": "VisDrone"
    }
]


# ============================================================
# READ NDJSON
# ============================================================

all_records = []

for dataset in DATASETS:

    print(f"\n📖 Reading {dataset['name']}...")

    with open(dataset["file"], "r", encoding="utf-8") as f:

        for line in f:

            try:
                data = json.loads(line)
            except json.JSONDecodeError:
                continue

            if data.get("type") == "image":

                all_records.append({
                    "data": data,
                    "dataset": dataset["name"]
                })

print("\n================================")
print("TOTAL IMAGE RECORDS:", len(all_records))
print("================================")


# ============================================================
# HELPER: GET IMAGE INFORMATION
# ============================================================

def get_image_info(data):

    image = data.get("image")

    if isinstance(image, dict):
        return image

    return data


# ============================================================
# PREPARE DOWNLOAD TASKS
# ============================================================

tasks = []

for item in all_records:

    data = item["data"]
    dataset_name = item["dataset"]

    image_info = get_image_info(data)

    # Find image URL
    url = (
        image_info.get("url")
        or data.get("url")
        or data.get("image_url")
    )

    # Find filename
    filename = (
        image_info.get("file")
        or image_info.get("filename")
        or data.get("file")
        or data.get("filename")
    )

    if not url or not filename:
        continue

    # Dataset split
    split = data.get("split", "train")

    if split not in ["train", "val", "test"]:
        split = "train"

    # Add dataset name to avoid duplicate filenames
    safe_filename = (
        dataset_name + "_" + Path(filename).name
    )

    image_path = (
        BASE / "images" / split / safe_filename
    )

    tasks.append({
        "url": url,
        "image_path": image_path,
        "data": data,
        "dataset": dataset_name,
        "split": split,
        "filename": safe_filename
    })


print("Images with downloadable URLs:", len(tasks))


# ============================================================
# DOWNLOAD FUNCTION
# ============================================================

def download_image(task):

    url = task["url"]
    path = task["image_path"]

    if path.exists():
        return True, None

    try:

        response = requests.get(
            url,
            timeout=60
        )

        response.raise_for_status()

        with open(path, "wb") as f:
            f.write(response.content)

        return True, None

    except Exception as e:

        return False, str(e)


# ============================================================
# DOWNLOAD IMAGES
# ============================================================

print("\n⬇️ DOWNLOADING IMAGES...")
print("This can take some time because the dataset is large.\n")

successful = 0
failed = 0
failed_tasks = []

with ThreadPoolExecutor(max_workers=16) as executor:

    futures = {
        executor.submit(download_image, task): task
        for task in tasks
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Downloading"
    ):

        success, error = future.result()

        task = futures[future]

        if success:
            successful += 1
        else:
            failed += 1
            failed_tasks.append(
                (task["url"], error)
            )


print("\n================================")
print("DOWNLOAD COMPLETE")
print("================================")
print("Successful:", successful)
print("Failed:", failed)


# ============================================================
# CREATE UNIFIED YOLO LABELS
# ============================================================

print("\n🏷️ Creating unified annotations...")

total_boxes = 0
class_counts = {name: 0 for name in CLASS_NAMES}

for task in tqdm(
    tasks,
    desc="Creating labels"
):

    image_path = task["image_path"]

    if not image_path.exists():
        continue

    data = task["data"]
    dataset_name = task["dataset"]

    annotations = data.get("annotations") or {}

    boxes = annotations.get("boxes") or []

    unified_boxes = []

    for box in boxes:

        if not box:
            continue

        original_class_id = int(box[0])

        # ----------------------------------------------------
        # SARD
        # ----------------------------------------------------

        if dataset_name == "SARD":

            # SARD class 0 = person
            if original_class_id == 0:

                new_class = 0

                unified_boxes.append([
                    new_class,
                    float(box[1]),
                    float(box[2]),
                    float(box[3]),
                    float(box[4])
                ])

                class_counts["person"] += 1


        # ----------------------------------------------------
        # VISDRONE
        # ----------------------------------------------------

        elif dataset_name == "VisDrone":

            # VisDrone:
            # 0 = pedestrian
            # 1 = people
            # 2 = bicycle
            # 3 = car
            # 4 = van
            # 5 = truck
            # 6 = tricycle
            # 7 = awning-tricycle
            # 8 = bus
            # 9 = motor

            VISDRONE_MAP = {
                0: 0,  # pedestrian -> person
                1: 0,  # people -> person
                2: 1,  # bicycle
                3: 2,  # car
                4: 3,  # van
                5: 4,  # truck
                6: 5,  # tricycle
                7: 6,  # awning-tricycle
                8: 7,  # bus
                9: 8   # motor
            }

            if original_class_id not in VISDRONE_MAP:
                continue

            new_class = VISDRONE_MAP[original_class_id]

            unified_boxes.append([
                new_class,
                float(box[1]),
                float(box[2]),
                float(box[3]),
                float(box[4])
            ])

            class_counts[
                CLASS_NAMES[new_class]
            ] += 1


    # --------------------------------------------------------
    # WRITE YOLO LABEL FILE
    # --------------------------------------------------------

    if unified_boxes:

        label_path = (
            BASE
            / "labels"
            / task["split"]
            / f"{Path(task['filename']).stem}.txt"
        )

        with open(label_path, "w") as f:

            for box in unified_boxes:

                f.write(
                    " ".join(
                        map(str, box)
                    )
                    + "\n"
                )

        total_boxes += len(unified_boxes)


# ============================================================
# CREATE data.yaml
# ============================================================

yaml_path = BASE / "data.yaml"

yaml_content = """path: /content/combined_object_detection

train: images/train
val: images/val
test: images/test

names:
  0: person
  1: bicycle
  2: car
  3: van
  4: truck
  5: tricycle
  6: awning-tricycle
  7: bus
  8: motor
"""

with open(yaml_path, "w") as f:
    f.write(yaml_content)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n\n==========================================")
print("       🎉 DATASET READY 🎉")
print("==========================================")

for split in ["train", "val", "test"]:

    image_count = len(
        list(
            (BASE / "images" / split).glob("*")
        )
    )

    label_count = len(
        list(
            (BASE / "labels" / split).glob("*.txt")
        )
    )

    print(
        f"{split.upper():5} : "
        f"{image_count} images | "
        f"{label_count} labels"
    )

print("\nTotal bounding boxes:", total_boxes)

print("\nClass distribution:")

for class_name, count in class_counts.items():

    print(
        f"{class_name:20} : {count}"
    )

print("\nDataset location:")
print(BASE)

print("\n✅ data.yaml created")

✅ Dataset folders created

📖 Reading SARD...

📖 Reading VisDrone...

TOTAL IMAGE RECORDS: 14376
Images with downloadable URLs: 14376

⬇️ DOWNLOADING IMAGES...
This can take some time because the dataset is large.



Downloading:   0%|          | 0/14376 [00:00<?, ?it/s]


DOWNLOAD COMPLETE
Successful: 14376
Failed: 0

🏷️ Creating unified annotations...


Creating labels:   0%|          | 0/14376 [00:00<?, ?it/s]



       🎉 DATASET READY 🎉
TRAIN : 10505 images | 9939 labels
VAL   : 1691 images | 1514 labels
TEST  : 2180 images | 2096 labels

Total bounding boxes: 464411

Class distribution:
person               : 155133
bicycle              : 13069
car                  : 186979
van                  : 32700
truck                : 16283
tricycle             : 6387
awning-tricycle      : 4374
bus                  : 9117
motor                : 40369

Dataset location:
/content/combined_object_detection

✅ data.yaml created


In [8]:
import shutil

zip_path = shutil.make_archive(
    "/content/combined_object_detection",
    "zip",
    "/content/combined_object_detection"
)

print("✅ ZIP created:")
print(zip_path)

✅ ZIP created:
/content/combined_object_detection.zip


In [10]:
from google.colab import files

files.download(
    "/content/combined_object_detection.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
with open("/content/combined_object_detection/data.yaml", "r") as f:
    print(f.read())

path: /content/combined_object_detection

train: images/train
val: images/val
test: images/test

names:
  0: person
  1: bicycle
  2: car
  3: van
  4: truck
  5: tricycle
  6: awning-tricycle
  7: bus
  8: motor

